In [2]:

import pandas as pd
import numpy as np

from collections import Counter, defaultdict
file_path = "words_pos.csv"

df = pd.read_csv(file_path)

print("=" * 60)
print("HMM POS TAGGER")
print("=" * 60)

print("\nDataset Shape:")
print(df.shape)

print("\nDataset Columns:")
print(df.columns.tolist())

print("\nFirst 5 Rows:")
print(df.head())

if "Unnamed: 0" in df.columns:
    df = df.drop(
        columns=["Unnamed: 0"]
    )

df = df.rename(
    columns={
        "word": "word",
        "pos_tag": "tag"
    }
)

if "word" not in df.columns:
    raise ValueError(
        "word column not found"
    )

if "tag" not in df.columns:
    raise ValueError(
        "pos_tag column not found"
    )

data = df[
    ["word", "tag"]
].dropna().copy()

data["word"] = (
    data["word"]
    .astype(str)
    .str.lower()
    .str.strip()
)

data["tag"] = (
    data["tag"]
    .astype(str)
    .str.upper()
    .str.strip()
)

data = data[
    data["word"] != ""
].reset_index(drop=True)


print("\nDataset after cleaning:")
print(data.shape)

print("\n" + "=" * 60)
print("DATASET SUMMARY")
print("=" * 60)

print(
    "Number of words:",
    len(data)
)

print(
    "Unique words:",
    data["word"].nunique()
)

print(
    "Unique POS tags:",
    data["tag"].nunique()
)

print(
    "\nPOS Tags:"
)

print(
    sorted(
        data["tag"].unique()
    )
)

tag_counts = Counter(
    data["tag"]
)

print("\n" + "=" * 60)
print("POS TAG COUNTS")
print("=" * 60)

print(tag_counts)

word_tag_counts = defaultdict(
    Counter
)

for word, tag in zip(
    data["word"],
    data["tag"]
):

    word_tag_counts[word][tag] += 1
emission_probabilities = defaultdict(
    dict
)

for word in word_tag_counts:

    total = sum(
        word_tag_counts[word].values()
    )

    for tag in word_tag_counts[word]:

        emission_probabilities[
            word
        ][tag] = (
            word_tag_counts[word][tag]
            / total
        )
word_best_tag = {}

for word in word_tag_counts:

    word_best_tag[word] = (
        word_tag_counts[word]
        .most_common(1)[0][0]
    )
print("\n" + "=" * 60)
print("SAMPLE EMISSION PROBABILITIES")
print("=" * 60)

sample_words = [
    "the",
    "dog",
    "runs",
    "love",
    "computer",
    "beautiful"
]

for word in sample_words:

    if word in emission_probabilities:

        print(
            word,
            "->",
            emission_probabilities[word]
        )

tags = sorted(
    data["tag"].unique()
)


transition_probabilities = defaultdict(
    dict
)
default_transition = 0.01
for previous_tag in tags:

    for current_tag in tags:

        transition_probabilities[
            previous_tag
        ][current_tag] = (
            default_transition
        )

transition_rules = {

    "DT": {
        "NN": 0.45,
        "NNS": 0.35,
        "JJ": 0.15,
        "VBN": 0.05
    },

    "JJ": {
        "NN": 0.55,
        "NNS": 0.25,
        "JJ": 0.10,
        "VBG": 0.05,
        "VBN": 0.05
    },

    "NN": {
        "VB": 0.25,
        "VBD": 0.15,
        "VBP": 0.20,
        "VBZ": 0.20,
        "VBG": 0.05,
        "IN": 0.10,
        "NN": 0.05
    },

    "NNS": {
        "VB": 0.20,
        "VBP": 0.30,
        "VBD": 0.10,
        "IN": 0.20,
        "NNS": 0.10,
        "JJ": 0.10
    },

    "PRP": {
        "VB": 0.20,
        "VBP": 0.35,
        "VBD": 0.20,
        "VBZ": 0.15,
        "MD": 0.10
    },

    "PRP$": {
        "NN": 0.55,
        "NNS": 0.30,
        "JJ": 0.15
    },

    "IN": {
        "DT": 0.30,
        "NN": 0.30,
        "NNS": 0.20,
        "JJ": 0.15,
        "PRP": 0.05
    },

    "TO": {
        "VB": 0.80,
        "NN": 0.10,
        "JJ": 0.10
    },

    "MD": {
        "VB": 0.70,
        "RB": 0.15,
        "VBN": 0.10,
        "VBG": 0.05
    },

    "VB": {
        "DT": 0.15,
        "NN": 0.20,
        "NNS": 0.10,
        "RB": 0.15,
        "IN": 0.15,
        "JJ": 0.10,
        "TO": 0.10,
        "VBG": 0.05
    },

    "VBD": {
        "DT": 0.15,
        "NN": 0.20,
        "NNS": 0.10,
        "RB": 0.15,
        "IN": 0.15,
        "JJ": 0.10,
        "VBN": 0.10,
        "VBG": 0.05
    },

    "VBP": {
        "DT": 0.15,
        "NN": 0.20,
        "NNS": 0.10,
        "RB": 0.15,
        "IN": 0.15,
        "JJ": 0.10,
        "VBG": 0.10,
        "VBN": 0.05
    },

    "VBZ": {
        "DT": 0.15,
        "NN": 0.20,
        "NNS": 0.10,
        "RB": 0.15,
        "IN": 0.15,
        "JJ": 0.10,
        "VBG": 0.10,
        "VBN": 0.05
    },

    "VBG": {
        "DT": 0.15,
        "NN": 0.20,
        "NNS": 0.10,
        "RB": 0.15,
        "IN": 0.15,
        "JJ": 0.10,
        "VBG": 0.05,
        "VBN": 0.05,
        "END": 0.10
    },

    "VBN": {
        "DT": 0.15,
        "NN": 0.20,
        "NNS": 0.10,
        "RB": 0.15,
        "IN": 0.15,
        "JJ": 0.10,
        "VBN": 0.05,
        "VBG": 0.05
    },

    "RB": {
        "VB": 0.20,
        "VBD": 0.15,
        "VBP": 0.15,
        "VBZ": 0.15,
        "JJ": 0.15,
        "RB": 0.10,
        "IN": 0.10
    }
}
for previous_tag in transition_rules:

    for current_tag in transition_rules[
        previous_tag
    ]:

        if current_tag in tags:

            transition_probabilities[
                previous_tag
            ][current_tag] = (
                transition_rules[
                    previous_tag
                ][current_tag]
            )

for previous_tag in tags:

    total = sum(
        transition_probabilities[
            previous_tag
        ].values()
    )

    if total > 0:

        for current_tag in tags:

            transition_probabilities[
                previous_tag
            ][current_tag] /= total
start_probabilities = {}

for tag in tags:

    start_probabilities[tag] = 0.01

for tag, value in {
    "DT": 0.30,
    "PRP": 0.20,
    "NN": 0.15,
    "NNS": 0.10,
    "JJ": 0.05,
    "RB": 0.05,
    "VB": 0.05,
    "VBD": 0.05,
    "IN": 0.05
}.items():

    if tag in start_probabilities:

        start_probabilities[tag] = value

total = sum(
    start_probabilities.values()
)

for tag in tags:

    start_probabilities[tag] /= total
def unknown_word_probabilities(word):
    probabilities = {}
    for tag in tags:
        probabilities[tag] = 1e-6
    if word in [
        "i",
        "you",
        "he",
        "she",
        "we",
        "they",
        "me",
        "him",
        "her",
        "us",
        "them"
    ]:

        if "PRP" in probabilities:
            probabilities["PRP"] = 0.9


    # Determiner
    elif word in [
        "a",
        "an",
        "the",
        "this",
        "that",
        "these",
        "those"
    ]:

        if "DT" in probabilities:
            probabilities["DT"] = 0.9
    elif word.endswith("ly"):

        if "RB" in probabilities:
            probabilities["RB"] = 0.9
    elif word.endswith("ing"):

        if "VBG" in probabilities:
            probabilities["VBG"] = 0.9
    elif word.endswith("ed"):

        if "VBD" in probabilities:
            probabilities["VBD"] = 0.45

        if "VBN" in probabilities:
            probabilities["VBN"] = 0.45
    elif word.endswith(
        (
            "ous",
            "ful",
            "able",
            "al",
            "ive",
            "less",
            "ic"
        )
    ):

        if "JJ" in probabilities:
            probabilities["JJ"] = 0.9
    elif word.endswith("s"):

        if "NNS" in probabilities:
            probabilities["NNS"] = 0.8

        if "NN" in probabilities:
            probabilities["NN"] = 0.1
    else:

        if "NN" in probabilities:
            probabilities["NN"] = 0.7

        if "JJ" in probabilities:
            probabilities["JJ"] = 0.1

        if "VB" in probabilities:
            probabilities["VB"] = 0.1

    total = sum(
        probabilities.values()
    )
    for tag in probabilities:
        probabilities[tag] /= total
    return probabilities
def get_emission_probabilities(
    word
):
    if word in emission_probabilities:
        probabilities = {}
        for tag in tags:
            probabilities[tag] = (
                emission_probabilities[
                    word
                ].get(
                    tag,
                    1e-6
                )
            )

        return probabilities
    return unknown_word_probabilities(
        word
    )

def viterbi(sentence):
    words = sentence.lower().split()
    if not words:
        return []
    n = len(words)
    viterbi_table = [
        {}
        for _ in range(n)
    ]
    backpointer = [
        {}
        for _ in range(n)
    ]
    first_word = words[0]

    emission = (
        get_emission_probabilities(
            first_word
        )
    )
    for tag in tags:

        start = start_probabilities.get(
            tag,
            1e-6
        )
        emit = emission.get(
            tag,
            1e-6
        )

        viterbi_table[0][tag] = (
            np.log(start)
            +
            np.log(emit)
        )

        backpointer[0][tag] = None

    for i in range(
        1,
        n
    ):
        word = words[i]

        emission = (
            get_emission_probabilities(
                word
            )
        )
        for current_tag in tags:
            emit = emission.get(
                current_tag,
                1e-6
            )
            best_score = -np.inf
            best_previous = None
            for previous_tag in tags:

                transition = (
                    transition_probabilities
                    .get(
                        previous_tag,
                        {}
                    )
                    .get(
                        current_tag,
                        1e-6
                    )
                )
                if transition <= 0:
                    transition = 1e-6
                score = (
                    viterbi_table[i - 1][
                        previous_tag
                    ]
                    +
                    np.log(transition)
                    +
                    np.log(emit)
                )
                if score > best_score:

                    best_score = score

                    best_previous = (
                        previous_tag
                    )


            viterbi_table[i][
                current_tag
            ] = best_score

            backpointer[i][
                current_tag
            ] = best_previous
    best_final_tag = max(
        viterbi_table[n - 1],
        key=viterbi_table[n - 1].get
    )
    best_tags = [
        best_final_tag
    ]

    for i in range(
        n - 1,
        0,
        -1
    ):

        previous = (
            backpointer[i][
                best_tags[-1]
            ]
        )

        best_tags.append(
            previous
        )


    best_tags.reverse()
    return list(
        zip(
            words,
            best_tags
        )
    )
print("\n" + "=" * 60)
print("HMM POS TAGGING TEST")
print("=" * 60)


test_sentence = (
    "the dog runs in the park"
)
test_result = viterbi(
    test_sentence
)
print("\nInput:")
print(test_sentence)
print("\nTagged Output:")
print(
    " ".join(
        [
            f"{word}/{tag}"
            for word, tag in test_result
        ]
    )
)
print("\n" + "=" * 60)
print("REAL-TIME HMM POS TAGGER")
print("=" * 60)


user_sentence = input(
    "\nEnter a sentence: "
)
user_result = viterbi(
    user_sentence
)
print("\nPOS Tagged Sentence:")


print(
    " ".join(
        [
            f"{word}/{tag}"
            for word, tag in user_result
        ]
    )
)
print("\n" + "=" * 60)
print("WORD-BY-WORD POS TAGGING")
print("=" * 60)


for word, tag in user_result:

    print(
        f"{word} -> {tag}"
    )


print("\n" + "=" * 60)
print("PROGRAM COMPLETED")
print("=" * 60)

HMM POS TAGGER

Dataset Shape:
(370100, 3)

Dataset Columns:
['Unnamed: 0', 'word', 'pos_tag']

First 5 Rows:
   Unnamed: 0    word pos_tag
0           0      aa      NN
1           1     aaa      NN
2           2     aah      NN
3           3   aahed     VBN
4           4  aahing     VBG

Dataset after cleaning:
(370100, 2)

DATASET SUMMARY
Number of words: 370100
Unique words: 370100
Unique POS tags: 25

POS Tags:
['CC', 'CD', 'DT', 'IN', 'JJ', 'JJR', 'JJS', 'MD', 'NN', 'NNS', 'PRP', 'PRP$', 'RB', 'RBR', 'TO', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'WDT', 'WP', 'WP$', 'WRB']

POS TAG COUNTS
Counter({'NN': 230960, 'NNS': 48541, 'JJ': 39206, 'RB': 17030, 'VBG': 16915, 'VBN': 14384, 'VB': 1810, 'VBD': 665, 'JJS': 353, 'IN': 91, 'DT': 31, 'VBZ': 30, 'PRP': 15, 'JJR': 13, 'CD': 12, 'MD': 12, 'PRP$': 7, 'CC': 5, 'WDT': 5, 'RBR': 4, 'WRB': 4, 'WP': 3, 'VBP': 2, 'TO': 1, 'WP$': 1})

SAMPLE EMISSION PROBABILITIES
the -> {'DT': 1.0}
dog -> {'NN': 1.0}
runs -> {'NNS': 1.0}
love -> {'NN': 1.0}


Enter a sentence:  i love nlp



POS Tagged Sentence:
i/NN love/NN nlp/NN

WORD-BY-WORD POS TAGGING
i -> NN
love -> NN
nlp -> NN

PROGRAM COMPLETED
